In [13]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

In [2]:
data = pd.read_csv("/Users/m.mohammed/Desktop/oil_daily.csv")
data["observation_date"] = pd.to_datetime(data["observation_date"])

## Baseline Models

### Naive (today = yesterday)

In [3]:
# Split by date
train = data[data["observation_date"] <= "2026-06-30"]
test = data[data["observation_date"] > "2026-06-30"]

print("Train rows:", len(train))
print("Test rows:", len(test))

Train rows: 1220
Test rows: 85


In [4]:
# Model 1: Naive (last training price)
naive_forecast = train["price"].iloc[-1]

In [5]:
# Model 2: Average (mean of training prices)
average_forecast = train["price"].mean()

In [6]:
print("Naive forecast:", naive_forecast)
print("Average forecast:", average_forecast)

Naive forecast: 70.56
Average forecast: 79.08901639344263


In [7]:
# Work out the MAE for each model
naive_total_error = 0
average_total_error = 0

for actual_price in test["price"]:
    naive_total_error = naive_total_error + abs(actual_price - naive_forecast)
    average_total_error = average_total_error + abs(actual_price - average_forecast)

naive_mae = naive_total_error / len(test)
average_mae = average_total_error / len(test)

print("Naive MAE:", naive_mae)
print("Average MAE:", average_mae)

Naive MAE: 19.684705882352944
Average MAE: 12.639434908389577


In [8]:
first_test_row = len(train)
last_test_row = len(data)

total_error = 0
count = 0

for i in range(first_test_row, last_test_row):
    actual_price = data["price"].iloc[i]
    forecast = data["price"].iloc[i - 1]
    total_error = total_error + abs(actual_price - forecast)
    count = count + 1

one_day_mae = total_error / count
print("One-day-ahead naive MAE:", one_day_mae)

One-day-ahead naive MAE: 2.404588235294118


### 5-day Moving Average

In [9]:
first_test_row = len(train)
last_test_row = len(data)

naive_total = 0
ma5_total = 0
count = 0

for i in range(first_test_row, last_test_row):
    actual_price = data["price"].iloc[i]

    # Naive: yesterday's price
    naive_forecast = data["price"].iloc[i - 1]

    # Moving average: average of the last 5 days
    last_5_total = 0
    for j in range(1, 6):
        last_5_total = last_5_total + data["price"].iloc[i - j]
    ma5_forecast = last_5_total / 5

    naive_total = naive_total + abs(actual_price - naive_forecast)
    ma5_total = ma5_total + abs(actual_price - ma5_forecast)
    count = count + 1

print("Naive MAE:", naive_total / count)
print("5-day moving average MAE:", ma5_total / count)

Naive MAE: 2.404588235294118
5-day moving average MAE: 3.611388235294118


### Exponential Smoothing 

In [11]:

train = data[data["observation_date"] <= "2026-06-30"]
first_test_row = len(train)
last_row = len(data)

alphas = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

for alpha in alphas:
    level = data["price"].iloc[0]

    train_error = 0
    train_count = 0
    test_error = 0
    test_count = 0

    for i in range(1, last_row):
        actual_price = data["price"].iloc[i]
        forecast = level
        error = abs(actual_price - forecast)

        if i < first_test_row:
            train_error = train_error + error
            train_count = train_count + 1
        else:
            test_error = test_error + error
            test_count = test_count + 1

        level = alpha * actual_price + (1 - alpha) * level

    print("Alpha:", alpha)
    print("  Train MAE:", train_error / train_count)
    print("  Test MAE:", test_error / test_count)

Alpha: 0.1
  Train MAE: 3.1233502770409745
  Test MAE: 4.725005770539612
Alpha: 0.3
  Train MAE: 2.062572219596474
  Test MAE: 3.31335384721275
Alpha: 0.5
  Train MAE: 1.7388481677229626
  Test MAE: 2.790079886589433
Alpha: 0.7
  Train MAE: 1.562854612676078
  Test MAE: 2.540253366007385
Alpha: 0.9
  Train MAE: 1.488630214060293
  Test MAE: 2.4258807684869526
Alpha: 1.0
  Train MAE: 1.4752420016406886
  Test MAE: 2.404588235294118


### ARIMA

In [12]:
train = data[data["observation_date"] <= "2026-06-30"].copy()

train["change"] = train["price"] - train["price"].shift(1)

for lag in range(1, 6):
    train["past_change"] = train["change"].shift(lag)
    correlation = train["change"].corr(train["past_change"])
    print("Lag", lag, "correlation:", correlation)

Lag 1 correlation: 0.0127119118764665
Lag 2 correlation: -0.0699075272242651
Lag 3 correlation: -0.015184860486252626
Lag 4 correlation: 0.03491107382240911
Lag 5 correlation: -0.02141754750794016


In [14]:
first_test_row = len(train)
last_row = len(data)

arima_total = 0
naive_total = 0
count = 0

for i in range(first_test_row, last_row):
    history = data["price"].iloc[0:i].values
    actual_price = data["price"].iloc[i]

    model = ARIMA(history, order=(2, 1, 0))
    result = model.fit()
    arima_forecast = result.forecast(steps=1)[0]

    naive_forecast = data["price"].iloc[i - 1]

    arima_total = arima_total + abs(actual_price - arima_forecast)
    naive_total = naive_total + abs(actual_price - naive_forecast)
    count = count + 1

print("Test days:", count)
print("ARIMA MAE:", arima_total / count)
print("Naive MAE:", naive_total / count)

Test days: 85
ARIMA MAE: 2.428331222600324
Naive MAE: 2.404588235294118


In [17]:
first_test_row = len(train)
last_row = len(data)

dates = []
actuals = []
naive_list = []
ma5_list = []
arima_list = []

naive_sq = 0
ma5_sq = 0
arima_sq = 0
count = 0

for i in range(first_test_row, last_row):
    actual_price = data["price"].iloc[i]

    naive_forecast = data["price"].iloc[i - 1]

    last_5_total = 0
    for j in range(1, 6):
        last_5_total = last_5_total + data["price"].iloc[i - j]
    ma5_forecast = last_5_total / 5

    history = data["price"].iloc[0:i].values
    model = ARIMA(history, order=(2, 1, 0))
    result = model.fit()
    arima_forecast = result.forecast(steps=1)[0]

    dates.append(data["observation_date"].iloc[i])
    actuals.append(actual_price)
    naive_list.append(naive_forecast)
    ma5_list.append(ma5_forecast)
    arima_list.append(arima_forecast)

    naive_sq = naive_sq + (actual_price - naive_forecast) ** 2
    ma5_sq = ma5_sq + (actual_price - ma5_forecast) ** 2
    arima_sq = arima_sq + (actual_price - arima_forecast) ** 2
    count = count + 1

print("Naive RMSE:", (naive_sq / count) ** 0.5)
print("5-day MA RMSE:", (ma5_sq / count) ** 0.5)
print("ARIMA RMSE:", (arima_sq / count) ** 0.5)

results = pd.DataFrame({
    "observation_date": dates,
    "actual_price": actuals,
    "naive_forecast": naive_list,
    "ma5_forecast": ma5_list,
    "arima_forecast": arima_list
})

results.to_csv("../data/forecast_results.csv", index=False)
print("Saved forecast_results.csv with", len(results), "rows")

Naive RMSE: 3.401324828391915
5-day MA RMSE: 4.445963562603725
ARIMA RMSE: 3.422921922766073
Saved forecast_results.csv with 85 rows
